# Per-token L2 across the note

**Previous:** [01_capture](01_capture.ipynb)  
**Home:** [../../00_START_HERE.ipynb](../../00_START_HERE.ipynb)  
**Next:** [03_layer_sweep](03_layer_sweep.ipynb)

**Kernel:** `CXR local Qwen (faiss_gpu1)`  
**Status:** executable (calls `scripts/` — same as CLI)

---

## 1. Why am I learning this?

Executable Observe lesson — same as `look.py --mode positions`.

## 2. Mental model

Read activations / logits. Do not write the forward.

## 3. Clinical question

What does this mode show on the built-in oncology prompts?

## 4. Prediction

Non-trivial numbers / strings; see CLI output style.


## 5. Minimal Python — setup + load (run once)


In [1]:
# Shared bootstrap — run this first in every executable notebook
import sys
from pathlib import Path

# notebooks/_lib regardless of how deep this notebook sits
_here = Path.cwd().resolve()
for _p in [_here, *_here.parents]:
    if (_p / "_lib" / "cxr_boot.py").is_file():
        sys.path.insert(0, str(_p / "_lib"))
        break
    if (_p / "notebooks" / "_lib" / "cxr_boot.py").is_file():
        sys.path.insert(0, str(_p / "notebooks" / "_lib"))
        break
else:
    raise RuntimeError("Cannot find notebooks/_lib/cxr_boot.py — open Jupyter with notebooks/ as root")

import cxr_boot
ctx = cxr_boot.setup(layer=20, max_new=24, load_model=False)
model, tok = ctx["model"], ctx["tok"]
NOTE, LAYER, MAX_NEW = ctx["NOTE"], ctx["LAYER"], ctx["MAX_NEW"]
PROMPT_A, PROMPT_B, PROMPT_TEST = ctx["PROMPT_A"], ctx["PROMPT_B"], ctx["PROMPT_TEST"]
look, intervene, process = ctx["look"], ctx["intervene"], ctx["process"]
print("NOTE:", NOTE)
print("LAYER:", LAYER, "ready")


Qwen already loaded on :8270 pid=762992 (Qwen/Qwen2.5-7B-Instruct) — not loading again.
look.cmd_capture(model, tok, LAYER, NOTE) will use that copy.
NOTE: Patient received FOLFOX. Disease progressed. FOLFOX was discontinued.
LAYER: 20 ready


## 6. Run


In [2]:
look.cmd_positions(model, tok, LAYER, NOTE)

layer 20  tokens=15  dim=3584
  [  0] L2=15568.237  Patient
  [  1] L2=134.829  Ġreceived
  [  2] L2=117.631  ĠF
  [  3] L2=119.446  OL
  [  4] L2=140.427  FOX
  [  5] L2=129.358  .
  [  6] L2=133.687  ĠDisease
  [  7] L2=142.629  Ġprogressed
  [  8] L2=128.677  .
  [  9] L2=111.908  ĠF
  [ 10] L2=112.350  OL
  [ 11] L2=123.718  FOX
  [ 12] L2=132.923  Ġwas
  [ 13] L2=130.808  Ġdiscontinued
  [ 14] L2=114.623  .  <- last
(used Qwen already loaded on :8270 pid=780195 — no second load)


Absolutely. Here's a version I'd put directly into your learning notes.

## What we just did: inspecting every token at Layer 20

### 1. We connected the notebook to the already-running model

We started with:

```python
ctx = cxr_boot.setup(
    layer=20,
    max_new=24,
    load_model=False
)
```

The important part is:

```python
load_model=False
```

The notebook therefore **did not request another Qwen load**. Your setup detected that Qwen was already available through the shared service on `:8270`.

You got:

```text
Qwen already loaded on :8270 ... — not loading again
```

Conceptually:

```text
Jupyter Notebook
      │
      │ experiment request
      ▼
Shared service :8270
      │
      ▼
Qwen2.5-7B-Instruct
(already loaded)
```

This matters because your 7B Qwen model is large. We don't want every notebook or GUI loading its own copy.

---

## 2. We gave Qwen this clinical note

```text
Patient received FOLFOX.
Disease progressed.
FOLFOX was discontinued.
```

Qwen's tokenizer broke that text into **15 tokens**.

You saw:

```text
[ 0] Patient
[ 1] Ġreceived
[ 2] ĠF
[ 3] OL
[ 4] FOX
[ 5] .
[ 6] ĠDisease
[ 7] Ġprogressed
[ 8] .
[ 9] ĠF
[10] OL
[11] FOX
[12] Ġwas
[13] Ġdiscontinued
[14] .
```

The `Ġ` is essentially tokenizer notation indicating that the token occurs after a space.

Also notice that:

```text
FOLFOX
```

isn't necessarily one token. It became:

```text
ĠF
OL
FOX
```

This is important in mechanistic interpretability because we often investigate **token positions**, not human words.

---

# 3. We ran `cmd_positions`

You executed:

```python
look.cmd_positions(model, tok, LAYER, NOTE)
```

with:

```python
LAYER = 20
```

The experiment captured the model's representation at **every token position at Layer 20**.

The output begins:

```text
layer 20  tokens=15  dim=3584
```

That tells us three things:

**Layer:** 20
**Sequence length:** 15 tokens
**Residual-stream width:** 3,584 dimensions

Conceptually, Layer 20 produced something like:

```text
                     3584 dimensions
                  <---------------->

Patient       → [x₁, x₂, x₃, ........ x₃₅₈₄]
received      → [x₁, x₂, x₃, ........ x₃₅₈₄]
F             → [x₁, x₂, x₃, ........ x₃₅₈₄]
OL            → [x₁, x₂, x₃, ........ x₃₅₈₄]
FOX           → [x₁, x₂, x₃, ........ x₃₅₈₄]
...
discontinued  → [x₁, x₂, x₃, ........ x₃₅₈₄]
.             → [x₁, x₂, x₃, ........ x₃₅₈₄]
```

So we're dealing with a tensor conceptually shaped:

$$
15 \times 3584
$$

for this sequence when ignoring the batch dimension.

---

# 4. What are those 3,584 numbers?

This is one of the most important concepts in your curriculum.

At Layer 20, each token position has a **3,584-dimensional residual-stream representation**.

For example, the final period might conceptually be:

```text
[
  0.75879,
 -2.01758,
  0.82471,
  0.63623,
 -0.30640,
 ...
  another 3579 values
]
```

Those 3,584 values collectively describe the model's internal state at that token position.

We should **not** interpret individual dimensions as things like:

```text
dimension 51 = FOLFOX
dimension 882 = progression
dimension 1902 = treatment failure
```

Representations are generally distributed, and individual coordinates do not automatically have such simple meanings.

That's one reason we later use things like:

* difference directions
* projections
* probes
* SAEs
* causal interventions

to investigate what information is represented.

---

# 5. Then we calculated the L2 norm

Instead of printing all 3,584 values for every token, `cmd_positions` summarizes each vector using its **L2 norm**.

For a vector:

$$
h=(h_1,h_2,\ldots,h_{3584})
$$

the L2 norm is:

$$
||h||_2 =
\sqrt{h_1^2+h_2^2+\cdots+h_{3584}^2}
$$

Think of it as the **length or magnitude of the vector**.

That's why you got:

```text
received        L2 = 134.829
F               L2 = 117.631
OL              L2 = 119.446
FOX             L2 = 140.427
Disease         L2 = 133.687
progressed      L2 = 142.629
...
discontinued    L2 = 130.808
.               L2 = 114.623
```

We're compressing:

```text
3584 numbers
     ↓
one magnitude
     ↓
L2 norm
```

That makes it much easier to compare positions.

---

# 6. Why does every token have a different Layer-20 representation?

Because by Layer 20 the representation at a position isn't merely an embedding of that isolated token.

For example:

```text
progressed
```

exists in the context:

```text
Patient received FOLFOX.
Disease progressed.
```

Through earlier transformer layers, information has been mixed and transformed through attention, MLPs, and residual additions.

So by Layer 20, the representation associated with `"progressed"` is contextual.

Likewise, the final:

```text
.
```

isn't simply represented as:

> "This is a period."

Its residual state can contain information accumulated from the preceding sequence.

This is especially important for causal-language-model analysis.

---

# 7. Why were we previously looking only at the final token?

Previously you ran:

```python
look.cmd_capture(model, tok, LAYER, NOTE)
```

and got:

```text
layer 20 last-token L2: 114.6233
```

That experiment deliberately selected:

```python
[0, -1, :]
```

Meaning:

```text
0     = first item in batch
-1    = final token
:     = all 3584 dimensions
```

So it examined only:

```text
[14] "." → 114.623
```

Now `cmd_positions()` showed **all positions**.

And sure enough:

```text
cmd_capture

last token = 114.6233
```

matches:

```text
cmd_positions

[14] "." = 114.623
```

That's a useful validation that the two tools agree.

---

# 8. But L2 magnitude does NOT tell us what the model represents

This is probably the most important warning to put in your notes.

Suppose:

```text
progressed       L2 = 142
discontinued     L2 = 131
```

We **cannot conclude**:

> The model understands progression more strongly than discontinuation.

And we cannot conclude:

> `"progressed"` is more important to the answer.

The norm only tells us the **magnitude of the representation vector**.

It does not tell us its direction, semantic content, or causal importance.

That's why your curriculum progresses from:

```text
MAGNITUDE
What is happening to the vector?
        │
        ▼
REPRESENTATION
What information/direction can we detect?
        │
        ▼
CAUSATION
Does changing/removing it alter behavior?
```

That's exactly the distinction you've already encountered with L20 versus L27.

---

# 9. There is an anomaly we should investigate

Your first position showed:

```text
[0] Patient
L2 = 15568.237
```

while virtually everything else was around:

```text
~110–143
```

That's extremely different.

Do **not** write in your notes:

> "Patient has an enormous representation."

Instead write:

> **Observation:** Position 0 has an anomalously large L2 norm (~15,568 versus ~110–143 elsewhere). The cause has not yet been established and should be checked before interpretation.

It could be an artifact of the capture path, position handling, model architecture, service serialization, or something else. We need to investigate it rather than attach a semantic explanation to it.

---

## The one-paragraph version for your notebook

You can put this directly into your notes:

> **Experiment — Layer-20 token-position inspection:** We passed the clinical note *“Patient received FOLFOX. Disease progressed. FOLFOX was discontinued.”* through Qwen2.5-7B-Instruct and captured the Layer-20 residual-stream representation at every token position. The tokenizer produced 15 tokens, and each position was represented by a 3,584-dimensional vector. `cmd_positions()` calculated the L2 norm of each vector, providing a simple measure of its magnitude. The final token had an L2 norm of 114.623, matching the earlier `cmd_capture()` last-token result and providing a consistency check between the two tools. These norms describe vector magnitude only; they do not establish semantic content or causal importance. Position 0 showed an anomalously large norm (~15,568), which should be investigated before being interpreted.

And the bigger lesson is:

> **We have gone from knowing that Layer 20 contains a 3,584-dimensional residual state to actually inspecting that state separately at every token position in the clinical note.**

The next conceptual question is no longer *"how big are these vectors?"* It becomes **"what information is represented in their directions?"** That is the bridge from basic activation inspection into actual representation analysis.


## 7–8. What happened / What did I learn?

_(fill after you run)_

## 9. Claim boundary

✓ We ran an Observe measurement.

✗ Observe ≠ causal proof. No intervene yet in this notebook.

## 10. CXR connection

`look.py --mode positions`

## 11. Questions

## 12. Revision notes

| Date | Change |
|------|--------|
| | |
